# Tables · the paper's benchmarks

The paper's tables, one block or column group per benchmark (the main text's Dark Key-to-Door, CountRecall and MazeRunner; the passive T-Maze v3 block with `placement = appendix`), from the saved records only:

- **Table 1** (endpoints and declared contrasts): every drawn cell's confirmation-roster endpoint with its joint seed/task interval and per-seed values, the summary-cleared level, the environment's cache-boundary intervention (attempt-cleared on Key-to-Door, current-token on CountRecall and the T-Maze, summary-cleared on MazeRunner), and the evicted-evidence column (Key-to-Door retained − attempt-cleared; CountRecall accuracy on the queries whose first evidence predates the open segment, every cell on the same queries; MazeRunner and T-Maze retained − summary-cleared). The declared contrasts of the study protocol with their dispositions at each environment's δ. Endpoints, interventions and contrasts are quoted from the confirmation tier reports, never recomputed.
- **The main-result table**: one row per main-text cell with the three main benchmarks' at-horizon endpoints, each benchmark's beyond-horizon reading from Figure 3's saved tables (Key-to-Door's last 500-call window and cumulative doors at 4,000 calls, CountRecall's retention within the trained stream (accuracy on the queries whose evidence is three rewrites old), MazeRunner's largest maze every seed has read) and the peak live state at 4,000 calls.
- **Table 2** (cost): the peak live inference state an actor holds within a task, at the trained horizon and at every extended horizon the environment has (maze sizes on MazeRunner), under one accounting for every cell, beside the bytes each cell allocated for its trained horizon; decision and write-boundary latency, FLOPs per decision, peak training memory, hours per GPU model (never pooled across models) and parameters.
- **Claims × environments matrix** (first appendix table): the paper's readings per benchmark as holds / inconclusive / descriptive / pending / not run, derived from the report dispositions where a rule exists and from Figure 3's and Figure 4's saved tables (H4 on Key-to-Door, the trained-stream retention on CountRecall, C3's continued pairs in the appendix, the larger-maze read on MazeRunner; no beyond-horizon axis on the T-Maze).
- **Table A1** (learning efficiency): the charged calls to each seed's first crossing, from Figure 4's saved tables.

Rules and declarations are in [EXPERIMENTS](../docs/EXPERIMENTS.md). Each table is written as CSV, Markdown and LaTeX with a notes file listing every record read; Table 1 is also split into main-text and appendix twins by placement.

In [ ]:
"""Locate the repository, the study roots and the output folders.

Environment overrides, for a copy of the records elsewhere:
  REASONED_ICRL_OUTPUTS   study roots (default <repo>/outputs)
  FIGURE_OUTPUT_ROOT      where the figures are written (default notebooks/outputs/figures);
                          the tables and notes go to the sibling data/ folder
"""
import os
import sys
from pathlib import Path

from IPython.display import Markdown, display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO / "notebooks"))

from figures_common import default_layout, markdown_preview, select_cells  # noqa: E402

OUTPUTS = Path(os.environ.get("REASONED_ICRL_OUTPUTS", REPO / "outputs"))
FIGURES_DIR = Path(
    os.environ.get("FIGURE_OUTPUT_ROOT", REPO / "notebooks/outputs/figures")
)
DATA_DIR = FIGURES_DIR.parent / "data"
layout = default_layout(REPO, outputs=OUTPUTS)
print(f"records under {OUTPUTS}\nfigures in  {FIGURES_DIR}\ntables to   {DATA_DIR}")

In [ ]:
import csv
import json
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from figures_common import (
    ABLATION_CONDITION,
    ALL_ENVIRONMENTS,
    APPENDIX_ENVIRONMENTS,
    APPENDIX_ONLY_CONDITIONS,
    DRAWN_CELLS,
    ENVIRONMENTS,
    METHOD_CONDITION,
    PAPER_ENVIRONMENTS,
    SEEDS,
    CellStyle,
    cell_state_rows,
    load_cell,
    stratify,
    write_csv,
    write_strata,
)

from reasoned_icrl.analysis.statistics import _bootstrap, _interval

HORIZONS = {
    "keydoor": (500, 1000, 2000, 4000),
    "countrecall": (103,),
    "mazerunner": (15, 17, 19, 21, 25),
    "tmaze": (128,),
}
"""Horizons at which Table 2 reports live state: the trained horizon first, then
every declared extension the environment has (CountRecall's live state is
reported on one pair; its continued pairs axis grows the carried state of the
growing cells only, Figure 3's saved tables carry it; MazeRunner's horizons are
maze sizes under the area-scaled budget; the T-Maze is read at its evaluation
corridor only)."""


WRITE_RULE = {"raw_summary": "RSM-O", "raw_summary_residual": "RSM-R"}
"""The RSM family's names in contrast labels, by the write rule's cell: RSM-R
is the residual write (the rewrite ablation), RSM-O
the replacing write and the paper's method."""


def rsm_name(condition):
    """``RSM-R`` / ``RSM-O`` for the family's cells, ``RSM`` for any other method."""
    return WRITE_RULE.get(condition, "RSM")


def contrast_family(method):
    """The declared contrast family (section 4) as (name, left, right, role), the method named by its
    write rule (RSM-O, RSM-R); the ablation's own family follows (RSM-R's). Rows are matched to the tier reports by the (left, right) cells,
    never by the report's row names."""
    me = rsm_name(method)
    rows = [
        (f"{me} - no carry", method, "raw_segment", "primary"),
        (f"full history - {me}", "full_context", method, "equivalence"),
        (f"{me} - Memo", method, "memo", "companion"),
        (f"{me} - Memo fixed", method, "memo_fixed", "companion"),
        ("Memo - Memo fixed", "memo", "memo_fixed", "companion"),
        (f"{me} - GRU", method, "full_gru", "companion"),
    ]
    if method != ABLATION_CONDITION:
        other = ABLATION_CONDITION
        it = rsm_name(other)
        rows += [
            (f"{me} - {it}", method, other, "write rule"),
            (f"{it} - no carry", other, "raw_segment", f"{it} family"),
            (f"full history - {it}", "full_context", other, f"{it} family"),
            (f"{it} - Memo", other, "memo", f"{it} family"),
            (f"{it} - Memo fixed", other, "memo_fixed", f"{it} family"),
            (f"{it} - GRU", other, "full_gru", f"{it} family"),
        ]
    return rows


def placement(*conditions, environment=None):
    """``appendix`` for a row that involves an appendix-only cell (Memo fixed,
    the segmentation control) or an appendix
    environment (the passive T-Maze), else
    ``main``."""
    if environment in APPENDIX_ENVIRONMENTS:
        return "appendix"
    return (
        "appendix" if any(c in APPENDIX_ONLY_CONDITIONS for c in conditions) else "main"
    )


FIGURE_TABLES = {
    "keydoor_horizon": "figure3_beyond_the_training_horizon_keydoor_contrasts.csv",
    "first_crossing": "figureA1_learning_matched_experience_first_crossing.csv",
    "keydoor_windows": "figure3_beyond_the_training_horizon_keydoor_windows.csv",
    "keydoor_state": "figure3_beyond_the_training_horizon_keydoor_state.csv",
    "countrecall_streams": "figure3_beyond_the_training_horizon_countrecall_streams.csv",
    "countrecall_retention": "figure3_beyond_the_training_horizon_countrecall_retention.csv",
    "countrecall_stream_contrasts": "figure3_beyond_the_training_horizon_countrecall_stream_contrasts.csv",
    "mazerunner_sizes": "figure3_beyond_the_training_horizon_mazerunner_sizes.csv",
}
"""Saved tables of the figure notebooks the matrix and Table A1 read when present."""


@dataclass(frozen=True)
class TableOptions:
    seeds: Sequence[int] = SEEDS
    cells: Sequence[CellStyle] = DRAWN_CELLS
    environments: Sequence[str] = ALL_ENVIRONMENTS
    """Every environment tabulated: the main text's three and the T-Maze, whose
    rows carry ``placement = appendix``."""
    split: str = "confirmation"
    method: str = METHOD_CONDITION
    """The paper's RSM (the residual rewrite): its panel
    fixes the CountRecall strata, the overwrite cell standing in while it is
    absent; the contrast family is read against it."""
    samples: int = 2000
    resampling_seed: int = 0
    horizons: dict = None  # type: ignore[assignment]

    def horizons_of(self, environment):
        return (self.horizons or HORIZONS)[environment]


# ------------------------------------------------------------------ reports
# The confirmation tier report of each environment (cells, contrasts,
# interventions, costs): the numbers Table 1 quotes.


def read_report(layout, environment, name, sources):
    """Rows of one report CSV, or an empty list when the report has none."""
    directory = layout.report_directory(environment)
    if directory is None:
        return []
    path = directory / f"{name}.csv"
    if not path.is_file():
        return []
    with path.open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    if rows and set(rows[0]) != {"status"}:
        sources.append(layout.relative(path))
        return rows
    return []


def _number(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _cell_row(rows, condition, history, rule="endpoint"):
    for r in rows:
        if (
            r.get("condition") == condition
            and r.get("history") == history
            and r.get("checkpoint_rule") == rule
        ):
            return r
    return None


def _intervention_row(rows, condition, dependence, rule="endpoint"):
    for r in rows:
        if (
            r.get("condition") == condition
            and r.get("dependence") == dependence
            and r.get("checkpoint_rule") == rule
        ):
            return r
    return None


# -------------------------------------------------------- evicted evidence
# CountRecall: accuracy on the queries whose first evidence predates the open
# segment (strata 1-3 of the retention axis), every cell on the same queries.


def countrecall_evicted(layout, contract, options, sources):
    """Per condition: exact accuracy over the queries in strata 1-3, one value
    per stream, with the joint seed/task interval and the opportunity count."""
    panels, staged = {}, []
    for cell in options.cells:
        events = load_cell(
            layout,
            "countrecall",
            cell.condition,
            options.split,
            "retained",
            contract=contract,
            seeds=options.seeds,
            sources=staged,
        )
        if events is not None:
            panels[cell.condition] = events
    reference = next(
        (
            c
            for c in dict.fromkeys(
                (options.method, METHOD_CONDITION, ABLATION_CONDITION)
            )
            if c in panels
        ),
        None,
    )
    if reference is None:
        return {}
    sources.extend(staged)
    strata = write_strata(panels[reference])
    rng = np.random.default_rng(options.resampling_seed)
    out = {}
    for condition, events in panels.items():
        grouped = stratify(events, strata)
        evicted = [e for stratum, rows in grouped.items() if stratum >= 1 for e in rows]
        per_task = {}
        for e in evicted:
            per_task.setdefault(
                (e.training_seed, e.task_id, e.rollout_seed), []
            ).append(e.numerator)
        seeds = [s for s in options.seeds if any(k[0] == s for k in per_task)]
        streams = sorted({(k[1], k[2]) for k in per_task})
        matrix = np.asarray(
            [
                [float(np.mean(per_task.get((s, t, r), [np.nan]))) for t, r in streams]
                for s in seeds
            ]
        )
        estimate = float(np.nanmean(matrix))
        lower, upper = _interval(
            _bootstrap(
                np.nan_to_num(matrix, nan=estimate), samples=options.samples, rng=rng
            ),
            estimate,
            0.95,
        )
        out[condition] = {
            "estimate": estimate,
            "lower": lower,
            "upper": upper,
            "per_seed": {
                str(s): float(np.nanmean(matrix[i])) for i, s in enumerate(seeds)
            },
            "opportunities": len(evicted) // max(len(seeds), 1),
        }
    return out


# ------------------------------------------------------------------ Table 1


def table1(layout, options, sources):
    """Per environment and drawn cell: the endpoint, the summary-cleared level,
    the environment's cache-boundary intervention level, their drops and the
    evicted-evidence column; per environment the declared contrasts."""
    endpoints, contrasts = [], []
    for environment in options.environments:
        spec = layout.spec(environment)
        cells = read_report(layout, environment, "cells", sources)
        interventions = read_report(layout, environment, "interventions", sources)
        declared = read_report(layout, environment, "contrasts", sources)
        evicted = (
            countrecall_evicted(
                layout, layout.contract("countrecall"), options, sources
            )
            if environment == "countrecall"
            else {}
        )
        for cell in options.cells:
            retained = _cell_row(cells, cell.condition, "retained")
            row = {
                "environment": environment,
                "condition": cell.condition,
                "label": cell.label,
                "placement": placement(cell.condition, environment=environment),
                "metric": spec.metric,
                "unit": spec.unit,
                "delta": spec.delta,
                "status": "complete" if retained else "no report row",
            }
            if retained:
                row.update(
                    estimate=_number(retained["estimate"]),
                    lower=_number(retained["lower"]),
                    upper=_number(retained["upper"]),
                    per_seed=retained["per_seed"],
                    tasks=_number(retained["tasks"]),
                )
                cleared = _cell_row(cells, cell.condition, "summary-cleared")
                drop = _intervention_row(
                    interventions, cell.condition, "retained - summary-cleared"
                )
                row.update(
                    summary_cleared=_number(cleared["estimate"]) if cleared else None,
                    summary_cleared_drop=_number(drop["estimate"]) if drop else None,
                    summary_cleared_drop_lower=_number(drop["lower"]) if drop else None,
                    summary_cleared_drop_upper=_number(drop["upper"]) if drop else None,
                )
                level = _cell_row(cells, cell.condition, spec.intervention)
                drop = _intervention_row(
                    interventions, cell.condition, f"retained - {spec.intervention}"
                )
                row.update(
                    intervention=spec.intervention,
                    intervention_level=_number(level["estimate"]) if level else None,
                    intervention_drop=_number(drop["estimate"]) if drop else None,
                    intervention_drop_lower=_number(drop["lower"]) if drop else None,
                    intervention_drop_upper=_number(drop["upper"]) if drop else None,
                )
                if environment == "keydoor":
                    source = _intervention_row(
                        interventions, cell.condition, "retained - attempt-cleared"
                    )
                    row.update(
                        evicted_evidence="retained - attempt-cleared",
                        evicted_estimate=_number(source["estimate"])
                        if source
                        else None,
                        evicted_lower=_number(source["lower"]) if source else None,
                        evicted_upper=_number(source["upper"]) if source else None,
                    )
                elif environment in ("tmaze", "mazerunner"):
                    source = _intervention_row(
                        interventions, cell.condition, "retained - summary-cleared"
                    )
                    row.update(
                        evicted_evidence="retained - summary-cleared",
                        evicted_estimate=_number(source["estimate"])
                        if source
                        else None,
                        evicted_lower=_number(source["lower"]) if source else None,
                        evicted_upper=_number(source["upper"]) if source else None,
                    )
                else:
                    source = evicted.get(cell.condition)
                    row.update(
                        evicted_evidence="accuracy on queries with pre-segment evidence",
                        evicted_estimate=source["estimate"] if source else None,
                        evicted_lower=source["lower"] if source else None,
                        evicted_upper=source["upper"] if source else None,
                        evicted_per_seed=", ".join(
                            f"{seed}: {value:.4f}"
                            for seed, value in source["per_seed"].items()
                        )
                        if source
                        else None,
                        evicted_opportunities=source["opportunities"]
                        if source
                        else None,
                    )
            endpoints.append(row)
        by_pair = {
            (r["left"], r["right"]): r
            for r in declared
            if r.get("checkpoint_rule") == "endpoint"
        }
        for name, left, right, role in contrast_family(options.method):
            r = by_pair.get((left, right))
            lower, upper = (
                (_number(r["lower"]), _number(r["upper"])) if r else (None, None)
            )
            contrasts.append(
                {
                    "environment": environment,
                    "contrast": name,
                    "role": role,
                    "placement": placement(left, right, environment=environment),
                    "left": left,
                    "right": right,
                    "delta": spec.delta,
                    "estimate": _number(r["estimate"]) if r else None,
                    "lower": lower,
                    "upper": upper,
                    "per_seed": r["per_seed"] if r else None,
                    "positive_seeds": _number(r["positive_seeds"]) if r else None,
                    "inside_delta": (lower >= -spec.delta and upper <= spec.delta)
                    if lower is not None and upper is not None
                    else None,
                    "disposition": r["disposition"] if r else "not in report",
                }
            )
    return endpoints, contrasts


# --------------------------------------------------------------- main table


def _ci(row, digits):
    """``estimate [lower, upper]`` at ``digits`` decimals, or ``pending``."""
    if row is None or row.get("estimate") in (None, ""):
        return "pending"
    estimate, lower, upper = (float(row[k]) for k in ("estimate", "lower", "upper"))
    return f"{estimate:.{digits}f} [{lower:.{digits}f}, {upper:.{digits}f}]"


MAIN_TABLE_HEADERS = {
    "cell": "Cell",
    "keydoor": "Key-to-Door doors, 500 calls",
    "countrecall": "CountRecall accuracy, 103 queries",
    "mazerunner": "MazeRunner goal fraction, 15 x 15",
    "last_window": "Key-to-Door doors, calls 3,501 to 4,000",
    "cumulative": "Key-to-Door cumulative doors, 4,000 calls",
    "last_pair": "CountRecall accuracy, evidence three rewrites old",
    "largest_maze": "MazeRunner goal fraction, largest maze",
    "state": "Peak live state (KB), 4,000 calls",
}
"""The main-result table's column headers, as the paper prints them: the
three main benchmarks at the training horizon, then each benchmark's
long-run reading (Key-to-Door's last window and cumulative doors beyond the
trained budget; CountRecall's retention within the trained stream, the
accuracy on the queries whose evidence is three C32 rewrites old; MazeRunner's
largest maze) and the peak live state."""


def _last_stratum(rows, condition):
    """The retained row of the oldest evidence stratum (three rewrites) for one
    cell, or ``None``."""
    mine = [
        r
        for r in rows
        if r["condition"] == condition and r.get("history", "retained") == "retained"
    ]
    if not mine:
        return None
    return max(mine, key=lambda r: int(r["stratum"]))


def _largest_maze(rows, condition, seeds):
    """Goal fraction at the largest size every seed of the cell has read: the
    seed mean and range (the joint interval is Figure 3's), or ``None``."""
    mine = [r for r in rows if r["condition"] == condition]
    complete = sorted(
        size
        for size in {int(r["size"]) for r in mine}
        if {int(r["seed"]) for r in mine if int(r["size"]) == size} >= set(seeds)
    )
    if not complete:
        return None
    size = complete[-1]
    values = [float(r["goal_fraction"]) for r in mine if int(r["size"]) == size]
    return {
        "size": size,
        "mean": sum(values) / len(values),
        "low": min(values),
        "high": max(values),
    }


def table_main(endpoints, options, data_dir):
    """The main-result table: one row per
    main-text cell; the three main benchmarks' endpoints at the training
    horizon (Table 1's estimates), then the beyond-horizon readings of Figure
    3 (Key-to-Door: completed doors in the last 500-call window of a
    4,000-call task and the cumulative doors; CountRecall: exact accuracy on the
    queries whose evidence is three rewrites old; MazeRunner: goal fraction at the largest
    maze every seed has read) and the peak live state per actor at 4,000
    calls."""
    ends = {(r["environment"], r["condition"]): r for r in endpoints}
    last = {}
    for r in _read_saved(data_dir, "keydoor_windows"):
        if r["condition"] not in last or int(r["window"]) > int(
            last[r["condition"]]["window"]
        ):
            last[r["condition"]] = r
    state = {r["condition"]: r for r in _read_saved(data_dir, "keydoor_state")}
    retention = _read_saved(data_dir, "countrecall_retention")
    sizes = _read_saved(data_dir, "mazerunner_sizes")
    digits = {"keydoor": 1, "countrecall": 3, "mazerunner": 3}
    heads = MAIN_TABLE_HEADERS
    rows = []
    for cell in options.cells:
        if placement(cell.condition) != "main":
            continue
        row = {heads["cell"]: cell.label, "condition": cell.condition}
        for environment in PAPER_ENVIRONMENTS:
            row[heads[environment]] = _ci(
                ends.get((environment, cell.condition)), digits[environment]
            )
        # The memoryless control is not run beyond the trained budget (Figure
        # 3 draws the memory carriers only): its long-run cells are blank by
        # design, not pending.
        beyond = cell.condition != "raw_segment"
        window = last.get(cell.condition)
        row[heads["last_window"]] = (
            _ci(window, 1) if window else ("pending" if beyond else "\u2014")
        )
        row[heads["cumulative"]] = (
            f"{float(window['cumulative_estimate']):.0f} "
            f"[{float(window['cumulative_lower']):.0f}, "
            f"{float(window['cumulative_upper']):.0f}]"
            if window
            else ("pending" if beyond else "\u2014")
        )
        # Figure 3 draws the memory carriers only, so the memoryless control has
        # no retention row by design.
        stratum = _last_stratum(retention, cell.condition)
        row[heads["last_pair"]] = (
            f"{_ci(stratum, 3)} (n = {stratum['queries']})"
            if stratum
            else ("pending" if beyond else "\u2014")
        )
        maze = _largest_maze(sizes, cell.condition, options.seeds)
        row[heads["largest_maze"]] = (
            f"{maze['mean']:.3f} ({maze['low']:.3f} to {maze['high']:.3f}, {maze['size']} x {maze['size']})"
            if maze
            else ("pending" if beyond else "\u2014")
        )
        measured = state.get(cell.condition)
        row[heads["state"]] = (
            f"{int(measured['peak_bytes']) / 1024:.0f}"
            if measured
            else ("pending" if beyond else "\u2014")
        )
        rows.append(row)
    return rows


# ------------------------------------------------------------------ Table 2


def table2(layout, options, sources):
    """Per environment and drawn cell: peak live state over a task at every
    horizon (one accounting for every cell) beside the bytes allocated for the
    trained horizon, decision and boundary latency, FLOPs per decision, peak
    training memory, parameters (from every seed's cost record) and hours per
    GPU model (from the report)."""
    rows = []
    for environment in options.environments:
        contract = layout.contract(environment)
        costs = read_report(layout, environment, "costs", sources)
        conditions = [c.condition for c in options.cells]
        states = {}
        for horizon in options.horizons_of(environment):
            for r in cell_state_rows(
                layout,
                environment,
                conditions,
                horizon,
                contract=contract,
                seeds=options.seeds,
                sources=sources,
            ):
                states.setdefault(r["condition"], {})[horizon] = (
                    r["bytes"],
                    r["method"],
                    r["allocated_bytes"],
                )
        for cell in options.cells:
            systems = []
            for seed in options.seeds:
                path = (
                    layout.run_directory(environment, cell.condition, seed)
                    / "systems.json"
                )
                if path.is_file():
                    systems.append(json.loads(path.read_text()))
            if not systems and cell.condition not in states:
                continue
            row = {
                "environment": environment,
                "condition": cell.condition,
                "label": cell.label,
                "seeds_measured": len(systems),
            }
            horizons = list(options.horizons_of(environment))
            mine = states.get(cell.condition, {})
            trained = mine.get(horizons[0])
            row["trained_horizon"] = horizons[0]
            row["peak_live_kb_trained"] = (
                round(trained[0] / 1e3, 1) if trained else None
            )
            row["allocated_kb_trained"] = (
                round(trained[2] / 1e3, 1) if trained else None
            )
            row["peak_live_kb_by_horizon"] = "; ".join(
                f"{h:,}: {mine[h][0] / 1e3:,.1f}" for h in horizons if h in mine
            )
            row["state_rule"] = "; ".join(sorted({v[1] for v in mine.values()}))

            def mean_of(key, systems=systems):
                values = [
                    s[key] for s in systems if isinstance(s.get(key), (int, float))
                ]
                return float(np.mean(values)) if values else None

            row.update(
                decision_latency_ms=(
                    mean_of("decision_latency_seconds") * 1e3
                    if mean_of("decision_latency_seconds") is not None
                    else None
                ),
                boundary_latency_ms=(
                    mean_of("boundary_latency_seconds") * 1e3
                    if mean_of("boundary_latency_seconds") is not None
                    else None
                ),
                decision_mflops=(
                    mean_of("decision_flops_counted") / 1e6
                    if mean_of("decision_flops_counted") is not None
                    else None
                ),
                peak_gpu_gb=(
                    mean_of("peak_gpu_bytes") / 1e9
                    if mean_of("peak_gpu_bytes") is not None
                    else None
                ),
                gradient_steps=mean_of("gradient_steps"),
                parameters=(
                    sum(systems[0]["parameters"].values())
                    if systems and isinstance(systems[0].get("parameters"), dict)
                    else None
                ),
            )
            hours = [
                f"{r['gpu']}: {r['training_hours']}"
                for r in costs
                if r.get("condition") == cell.condition
            ]
            row["hours_per_gpu_model"] = "; ".join(hours)
            rows.append(row)
    return rows


# --------------------------------------------------------- state memory
# The bounded carriers' selling point, as a table rather than a panel: per environment and horizon, one row for the state an
# actor holds on average over the task's records and one for the peak it holds
# within the task, one column per cell. One growth rule per cell and two
# readings of it (figures_common): Memo its recorded per-slot schedule, the
# full history one cache slot per record, every other cell its fixed buffer.


def format_bytes(value):
    """Bytes as KB or MB, the unit a reader compares at a glance."""
    if value is None:
        return "—"
    if value >= 1e6:
        return f"{value / 1e6:,.2f} MB"
    return f"{value / 1e3:,.1f} KB"


def state_memory(layout, options, sources):
    """Two rows per environment and horizon — the average over the task's
    records and the peak within the task — with one column per cell, plus the
    raw bytes and the rule behind every number in the long table."""
    table, long_rows = [], []
    for environment in options.environments:
        spec = layout.spec(environment)
        contract = layout.contract(environment)
        conditions = [c.condition for c in options.cells]
        for horizon in options.horizons_of(environment):
            rows = cell_state_rows(
                layout,
                environment,
                conditions,
                horizon,
                contract=contract,
                seeds=options.seeds,
                sources=sources,
            )
            by_condition = {r["condition"]: r for r in rows}
            # An environment whose cost records do not exist yet keeps its two
            # rows with an em dash per cell, so the table lists every
            # benchmark and says where the measurement is still missing.
            records = (
                next(iter(by_condition.values()))["records"]
                if by_condition
                else spec.records_at(horizon, contract)
            )
            for accounting, key in (("average", "mean_bytes"), ("peak", "peak_bytes")):
                entry = {
                    "environment": spec.title,
                    "horizon": horizon,
                    "horizon_unit": HORIZON_UNITS[spec.horizon_kind],
                    "records": records,
                    "state": accounting,
                    "placement": "appendix"
                    if environment in APPENDIX_ENVIRONMENTS
                    else "main",
                }
                for cell in options.cells:
                    row = by_condition.get(cell.condition)
                    entry[cell.label] = format_bytes(row[key] if row else None)
                table.append(entry)
            for cell in options.cells:
                row = by_condition.get(cell.condition)
                if row is None:
                    continue
                long_rows.append(
                    {
                        "environment": environment,
                        "condition": cell.condition,
                        "horizon": horizon,
                        "records": row["records"],
                        "mean_bytes": row["mean_bytes"],
                        "peak_bytes": row["peak_bytes"],
                        "allocated_bytes": row["allocated_bytes"],
                        "seeds_measured": row["seeds_measured"],
                        "rule": row["method"],
                    }
                )
    return table, long_rows


HORIZON_UNITS = {
    "calls": "charged calls",
    "stream": "records in the stream",
    "corridor": "corridor length",
    "maze": "maze size",
}
"""What a horizon counts, per environment kind, for the state table's column."""


# ------------------------------------------------------------ claims matrix


def _read_saved(data_dir, key):
    """Rows of one saved figure table, or ``[]`` when it is absent or was
    written empty (a lone ``status`` column)."""
    path = Path(data_dir) / FIGURE_TABLES[key]
    if not path.is_file():
        return []
    with path.open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    if rows and set(rows[0]) == {"status"}:
        return []
    return rows


def claims_matrix(endpoints, contrasts, options, data_dir):
    """One row per claim and environment: holds / inconclusive / descriptive /
    pending / not run, from the report dispositions where a reading rule exists
    (the contrast family and the interventions) and from the saved tables of
    Figures 3 and 4 (retention beyond the horizon, learning at matched experience)."""
    by = {(r["environment"], r["contrast"]): r for r in contrasts}
    ends = {(r["environment"], r["condition"]): r for r in endpoints}
    horizon = _read_saved(data_dir, "keydoor_horizon")
    crossings = _read_saved(data_dir, "first_crossing")
    stream_contrasts = _read_saved(data_dir, "countrecall_stream_contrasts")
    stream_rows = _read_saved(data_dir, "countrecall_streams")
    maze_sizes = _read_saved(data_dir, "mazerunner_sizes")
    rows = []

    def add(claim, environment, reading, basis):
        rows.append(
            {
                "claim": claim,
                "environment": environment,
                "reading": reading,
                "basis": basis,
            }
        )

    me = rsm_name(options.method)
    for environment in options.environments:
        spec = ENVIRONMENTS[environment]
        carry = by.get((environment, f"{me} - no carry"))
        if carry is None or carry["estimate"] is None:
            add(
                f"carry ({me} - no carry >= delta)",
                environment,
                "pending",
                "no report row",
            )
        else:
            add(
                f"carry ({me} - no carry >= delta)",
                environment,
                "holds"
                if carry["disposition"] == "consistent practical gain"
                else "inconclusive",
                f"{carry['disposition']}; {carry['estimate']:+.3f} [{carry['lower']:+.3f}, {carry['upper']:+.3f}]",
            )
        for claim, name in (
            (
                "rewriting matches the full history (inside ±delta)",
                f"full history - {me}",
            ),
            (
                f"rewriting matches accumulation ({me} - Memo inside ±delta)",
                f"{me} - Memo",
            ),
            (
                f"matched segmentation ({me} - Memo fixed inside ±delta)",
                f"{me} - Memo fixed",
            ),
            (f"bounded recurrence ({me} - GRU)", f"{me} - GRU"),
        ):
            r = by.get((environment, name))
            if r is None or r["estimate"] is None:
                add(claim, environment, "pending", "no report row")
                continue
            if name == f"{me} - GRU":
                reading = (
                    "RSM ahead"
                    if r["disposition"] == "consistent practical gain"
                    else "GRU ahead"
                    if (
                        r["upper"] is not None
                        and r["upper"] < 0
                        and r["estimate"] <= -spec.delta
                    )
                    else "inconclusive"
                )
            else:
                reading = "holds" if r["inside_delta"] else "inconclusive"
            basis = (
                f"{r['disposition']}; "
                if name == f"{me} - GRU"
                else ("inside" if r["inside_delta"] else "not inside")
                + f" ±{spec.delta:g}; "
            )
            add(
                claim,
                environment,
                reading,
                basis + f"{r['estimate']:+.3f} [{r['lower']:+.3f}, {r['upper']:+.3f}]",
            )
        it = rsm_name(ABLATION_CONDITION)
        ablation = by.get((environment, f"{me} - {it}"))
        if ablation is None or ablation["estimate"] is None:
            add(
                f"write rule ({me} - {it})",
                environment,
                "pending" if ablation else "not run",
                ablation["disposition"] if ablation else f"the method is {it}",
            )
        else:
            add(
                f"write rule ({me} - {it})",
                environment,
                ablation["disposition"],
                f"{ablation['estimate']:+.3f} [{ablation['lower']:+.3f}, {ablation['upper']:+.3f}]",
            )
        rsm = ends.get((environment, options.method))
        if rsm and rsm.get("summary_cleared_drop") is not None:
            lower = rsm.get("summary_cleared_drop_lower")
            add(
                "intervention (clearing the summary costs at least delta/2)",
                environment,
                "holds"
                if rsm["summary_cleared_drop"] >= spec.delta / 2 and (lower or 0) > 0
                else "inconclusive",
                f"retained - summary-cleared {rsm['summary_cleared_drop']:+.3f} [{lower:+.3f}, {rsm['summary_cleared_drop_upper']:+.3f}]",
            )
        else:
            add(
                "intervention (clearing the summary costs at least delta/2)",
                environment,
                "pending",
                "no intervention row",
            )
        if environment == "keydoor":
            h4a = [r for r in horizon if r.get("contrast", "").startswith("H4a")]
            h4b = [r for r in horizon if r.get("contrast", "").startswith("H4b")]
            if not h4a:
                add(
                    "retention beyond the horizon (H4a: RSM holds its rate)",
                    environment,
                    "pending",
                    "run Figure 3",
                )
                add(
                    "retention beyond the horizon (H4b: accumulation degrades)",
                    environment,
                    "pending",
                    "run Figure 3",
                )
            else:
                (a,) = h4a
                add(
                    "retention beyond the horizon (H4a: RSM holds its rate)",
                    environment,
                    "holds"
                    if a.get("within_delta_every_seed") == "True"
                    or a.get("no_fall_every_seed") == "True"
                    else "inconclusive",
                    f"{a['contrast']} {float(a['estimate']):+.2f} [{float(a['lower']):+.2f}, {float(a['upper']):+.2f}]",
                )
                add(
                    "retention beyond the horizon (H4b: accumulation degrades)",
                    environment,
                    "holds"
                    if h4b and all(r.get("consistent_gain") == "True" for r in h4b)
                    else "inconclusive",
                    "; ".join(
                        f"{r['contrast']} {float(r['estimate']):+.2f}" for r in h4b
                    ),
                )
        elif environment == "countrecall":
            retained = [
                r
                for r in _read_saved(data_dir, "countrecall_retention")
                if r.get("history", "retained") == "retained" and int(r["stratum"]) == 3
            ]
            if retained:
                by_cell = {r["condition"]: r for r in retained}
                mine_r = by_cell.get(options.method)
                others = {
                    c: by_cell[c]
                    for c in ("full_context", "memo", "full_gru")
                    if c in by_cell
                }
                add(
                    "retention within the trained stream (accuracy on evidence three rewrites old)",
                    environment,
                    "descriptive"
                    if mine_r is None or not others
                    else "RSM at or above every comparator"
                    if all(
                        float(mine_r["estimate"]) >= float(r["estimate"]) - spec.delta
                        for r in others.values()
                    )
                    else "a comparator ahead",
                    "; ".join(
                        f"{c} {float(r['estimate']):.3f} [{float(r['lower']):.3f}, {float(r['upper']):.3f}]"
                        for c, r in by_cell.items()
                    ),
                )
            else:
                add(
                    "retention within the trained stream (accuracy on evidence three rewrites old)",
                    environment,
                    "pending",
                    "run Figure 3",
                )
            mine = [r for r in stream_contrasts if r.get("contrast")]
            last = max((int(r["pair"]) for r in stream_rows), default=None)
            if not mine or last is None:
                add(
                    "continued pairs (C3), appendix (H9a: RSM holds its accuracy over the continued pairs)",
                    environment,
                    "pending",
                    "run Figure 3 (the C3 continued panels)",
                )
                add(
                    "continued pairs (C3), appendix (H9b: the full history and Memo fall below RSM)",
                    environment,
                    "pending",
                    "run Figure 3 (the C3 continued panels)",
                )
            else:
                me_rows = [
                    r
                    for r in stream_rows
                    if r["condition"] == options.method
                    and r.get("history", "retained") == "retained"
                ]
                first = next(
                    (float(r["estimate"]) for r in me_rows if int(r["pair"]) == 1), None
                )
                later = [float(r["estimate"]) for r in me_rows if int(r["pair"]) > 1]
                add(
                    "continued pairs (C3), appendix (H9a: RSM holds its accuracy over the continued pairs)",
                    environment,
                    "holds"
                    if first is not None and later and min(later) >= first - spec.delta
                    else "inconclusive"
                    if me_rows
                    else "pending",
                    f"{me} pair 1 {first:.3f}, pairs 2-{last} min {min(later):.3f}"
                    if first is not None and later
                    else "no method rows",
                )
                at_last = {r["contrast"]: r for r in mine if int(r["pair"]) == last}
                wanted = [f"{me} - full history", f"{me} - Memo"]
                found = [at_last[w] for w in wanted if w in at_last]
                add(
                    "continued pairs (C3), appendix (H9b: the full history and Memo fall below RSM)",
                    environment,
                    "holds"
                    if found
                    and len(found) == len(wanted)
                    and all(
                        float(r["lower"]) > 0 and float(r["estimate"]) >= spec.delta
                        for r in found
                    )
                    else "inconclusive"
                    if found
                    else "pending",
                    "; ".join(
                        f"{r['contrast']} at pair {last} {float(r['estimate']):+.3f} [{float(r['lower']):+.3f}, {float(r['upper']):+.3f}]"
                        for r in found
                    )
                    or "no contrast rows",
                )
        elif environment == "mazerunner":
            if not maze_sizes:
                add(
                    "adaptation beyond the horizon (H8a: RSM above the full history and Memo on larger mazes)",
                    environment,
                    "pending",
                    "run Figure 3 on the larger-maze panels",
                )
            else:
                per = {}
                for r in maze_sizes:
                    per.setdefault(r["condition"], {}).setdefault(
                        int(r["size"]), []
                    ).append(float(r["goal_fraction"]))
                summary = "; ".join(
                    f"{c} "
                    + "/".join(f"{sum(v) / len(v):.2f}" for _, v in sorted(by.items()))
                    for c, by in sorted(per.items())
                )
                add(
                    "adaptation beyond the horizon (H8a: RSM above the full history and Memo on larger mazes)",
                    environment,
                    "descriptive (the paired read is scripts/maze_read.py)",
                    f"seed-mean goal fraction by size: {summary}",
                )
        else:
            add(
                "retention beyond the horizon",
                environment,
                "not run",
                "no beyond-horizon axis is declared for this environment",
            )
        mine = [r for r in crossings if r.get("environment") == environment]
        if not mine:
            add(
                "learning at matched experience (RSM crosses before Memo on every seed)",
                environment,
                "pending",
                "run Figure 4",
            )
        else:
            primary = [
                r for r in mine if r.get("rule", "first") in ("first", "sustained")
            ]
            thresholds = sorted({r["threshold"] for r in primary})
            primary = (
                [r for r in primary if r["threshold"] == thresholds[-1]]
                if environment == "tmaze"
                else primary
            )
            rsm_c = {
                r["seed"]: r["first_crossing_calls"]
                for r in primary
                if r["condition"] == options.method
            }
            memo_c = {
                r["seed"]: r["first_crossing_calls"]
                for r in primary
                if r["condition"] == "memo"
            }
            if rsm_c and memo_c:
                earlier = all(
                    rsm_c[s] not in ("", None)
                    and (
                        memo_c.get(s) in ("", None)
                        or float(rsm_c[s]) < float(memo_c[s])
                    )
                    for s in rsm_c
                )
                add(
                    "learning at matched experience (RSM crosses before Memo on every seed)",
                    environment,
                    "holds (descriptive)" if earlier else "does not hold (descriptive)",
                    f"RSM {rsm_c}; Memo {memo_c}",
                )
            else:
                add(
                    "learning at matched experience (RSM crosses before Memo on every seed)",
                    environment,
                    "pending",
                    "cells missing in Figure 4's table",
                )
        add(
            "truncated gradients (RSM detach)",
            environment,
            "not run",
            "planned ablation, unallocated",
        )
        add(
            "probes (what the summary carries)",
            environment,
            "pending",
            "state dumps and probes not built",
        )
    return rows


def table_a1(data_dir, options):
    """Charged calls to the first crossing per environment, cell and seed, from
    Figure 4's saved table (millions of calls; censored seeds say so)."""
    rows = _read_saved(data_dir, "first_crossing")
    out = []
    for environment in options.environments:
        for cell in options.cells:
            mine = [
                r
                for r in rows
                if r["environment"] == environment and r["condition"] == cell.condition
            ]
            if not mine:
                continue
            for threshold in sorted({r["threshold"] for r in mine}, reverse=True):
                at = [r for r in mine if r["threshold"] == threshold]
                row = {
                    "environment": environment,
                    "condition": cell.condition,
                    "threshold": float(threshold),
                    "rule": at[0].get("rule", "first"),
                }
                for r in at:
                    calls = r["first_crossing_calls"]
                    row[f"seed_{r['seed']}_millions"] = (
                        round(float(calls) / 1e6, 3)
                        if calls not in ("", None)
                        else "censored"
                    )
                out.append(row)
    return out


# ------------------------------------------------------------------ writing


def _fmt(value):
    if value is None or value == "":
        return ""
    if isinstance(value, float):
        return f"{value:.3f}" if abs(value) < 100 else f"{value:,.0f}"
    return str(value)


def markdown_table(rows, columns=None):
    if not rows:
        return "_empty_"
    columns = columns or list(dict.fromkeys(k for r in rows for k in r))
    lines = ["| " + " | ".join(columns) + " |", "|" + "---|" * len(columns)]
    lines += ["| " + " | ".join(_fmt(r.get(c)) for c in columns) + " |" for r in rows]
    return "\n".join(lines)


def latex_table(rows, columns=None, caption=""):
    if not rows:
        return "% empty table\n"
    columns = columns or list(dict.fromkeys(k for r in rows for k in r))

    def escape(value):
        return str(value).replace("_", "\\_").replace("%", "\\%").replace("&", "\\&")

    head = " & ".join(escape(c) for c in columns) + " \\\\"
    body = [" & ".join(escape(_fmt(r.get(c))) for c in columns) + " \\\\" for r in rows]
    return "\n".join(
        [
            "\\begin{table}[t]",
            "\\centering\\small",
            f"\\caption{{{caption}}}",
            "\\begin{tabular}{" + "l" * len(columns) + "}",
            "\\toprule",
            head,
            "\\midrule",
            *body,
            "\\bottomrule",
            "\\end{tabular}",
            "\\end{table}",
            "",
        ]
    )


STATE_TABLE_HEAD = ["environment", "horizon", "horizon_unit", "records", "state"]
"""The state table's fixed columns; the cells follow in the drawn order."""

TABLE_COLUMNS = {
    "table1_endpoints": [
        "environment",
        "condition",
        "placement",
        "estimate",
        "lower",
        "upper",
        "per_seed",
        "summary_cleared",
        "intervention",
        "intervention_level",
        "evicted_evidence",
        "evicted_estimate",
        "evicted_lower",
        "evicted_upper",
        "evicted_opportunities",
        "status",
    ],
    "table1_contrasts": [
        "environment",
        "contrast",
        "role",
        "placement",
        "delta",
        "estimate",
        "lower",
        "upper",
        "per_seed",
        "positive_seeds",
        "inside_delta",
        "disposition",
    ],
    "table_main": [
        MAIN_TABLE_HEADERS[k]
        for k in (
            "cell",
            "keydoor",
            "countrecall",
            "mazerunner",
            "last_window",
            "cumulative",
            "last_pair",
            "largest_maze",
            "state",
        )
    ],
    "table2_costs": None,
    "claims_matrix": ["claim", "environment", "reading", "basis"],
    "table_a1_learning": None,
}


def build_tables(layout, options, data_dir):
    """Every table from the reports and records under ``layout`` (and the saved
    figure tables under ``data_dir`` where they exist), with the notes."""
    sources = []
    endpoints, contrasts = table1(layout, options, sources)
    state_table, state_long = state_memory(layout, options, sources)
    tables = {
        "table1_endpoints": endpoints,
        "table1_contrasts": contrasts,
        "table_main": table_main(endpoints, options, data_dir),
        "table_state_memory": state_table,
        "table_state_memory_bytes": state_long,
        "table2_costs": table2(layout, options, sources),
        "claims_matrix": claims_matrix(endpoints, contrasts, options, data_dir),
        "table_a1_learning": table_a1(data_dir, options),
    }
    notes = {
        "environments": list(options.environments),
        "cells": [c.condition for c in options.cells],
        "split": options.split,
        "method": options.method,
        "reports": {
            env: str(layout.report_directory(env)) for env in options.environments
        },
        "report_rows_found": {
            env: sorted(
                {
                    r["condition"]
                    for r in endpoints
                    if r["environment"] == env and r["status"] == "complete"
                }
            )
            for env in options.environments
        },
        "figure_tables_read": {
            k: (Path(data_dir) / v).is_file() for k, v in FIGURE_TABLES.items()
        },
        "horizons": {
            env: list(options.horizons_of(env)) for env in options.environments
        },
        "state_memory": "one average row and one peak row per environment and "
        "horizon, one column per cell; the bytes and the growth rule behind "
        "every cell are in table_state_memory_bytes",
        "samples": options.samples,
        "resampling_seed": options.resampling_seed,
    }
    return tables, notes, sorted(set(sources))


def save_tables(tables, notes, sources, data_dir, name="tables_three_benchmarks"):
    """CSV, Markdown and LaTeX per table plus the notes, under ``data_dir``."""
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for table, rows in tables.items():
        columns = TABLE_COLUMNS.get(table)
        if table == "table_state_memory" and rows:
            columns = STATE_TABLE_HEAD + [
                c for c in rows[0] if c not in (*STATE_TABLE_HEAD, "placement")
            ]
        base = data_dir / f"{name}_{table}"
        write_csv(base.with_suffix(".csv"), rows)
        base.with_suffix(".md").write_text(markdown_table(rows, columns) + "\n")
        base.with_suffix(".tex").write_text(
            latex_table(rows, columns, caption=table.replace("_", " "))
        )
        written += [base.with_suffix(s) for s in (".csv", ".md", ".tex")]
        if rows and "placement" in rows[0]:
            # Table 1 is split by placement: the main text's rows (the RSM
            # family, no carry, the published comparators) and the appendix's
            # (every row involving Memo fixed, the segmentation control).
            for where in ("main", "appendix"):
                subset = [r for r in rows if r["placement"] == where]
                path = data_dir / f"{name}_{table}_{where}.tex"
                path.write_text(
                    latex_table(
                        subset, columns, caption=f"{table.replace('_', ' ')} ({where})"
                    )
                )
                written.append(path)
    path = data_dir / f"{name}_notes.json"
    path.write_text(json.dumps({"notes": notes, "sources": sources}, indent=2) + "\n")
    written.append(path)
    return written

In [ ]:
# Parameters: `cells` restricts the tabulated cells (fixed order; all seven here,
# each row carrying its `placement`: Memo fixed's rows and contrasts go to the
# appendix), `method` is RSM-O (the replacing write; its panel fixes the
# CountRecall strata and the contrast family is read against it, RSM-R's family
# following), `samples` the bootstrap draw count of the evicted-evidence column
# (2000 in the paper). The environments are the main text's three (Key-to-Door,
# CountRecall, MazeRunner) and the T-Maze v3 block, whose rows carry
# placement = appendix.
options = TableOptions(samples=2000, resampling_seed=0, cells=select_cells())
tables, notes, sources = build_tables(layout, options, DATA_DIR)
print(json.dumps(notes["report_rows_found"], indent=1))
print("figure tables read:", notes["figure_tables_read"])

In [ ]:
# Table 1: endpoints and the declared contrasts, one block per benchmark.
display(
    Markdown(
        markdown_table(tables["table1_endpoints"], TABLE_COLUMNS["table1_endpoints"])
    )
)
display(
    Markdown(
        markdown_table(tables["table1_contrasts"], TABLE_COLUMNS["table1_contrasts"])
    )
)

In [ ]:
# The main-result table (one row per main-text cell: the three main benchmarks'
# at-horizon endpoints, each benchmark's beyond-horizon reading and the live
# state), then Table 2 (cost), the claims x environments matrix and Table A1.
display(Markdown(markdown_table(tables["table_main"], TABLE_COLUMNS["table_main"])))
# The state a bounded carrier holds: one average row and one peak row per
# environment and horizon, one column per cell.
display(
    Markdown(
        markdown_table(
            tables["table_state_memory"],
            STATE_TABLE_HEAD + [c.label for c in options.cells],
        )
    )
)
display(Markdown(markdown_preview(tables["table2_costs"], limit=24)))
display(
    Markdown(markdown_table(tables["claims_matrix"], TABLE_COLUMNS["claims_matrix"]))
)
display(Markdown(markdown_preview(tables["table_a1_learning"], limit=24)))

In [ ]:
# Save: CSV, Markdown and LaTeX per table and the notes (every record read) under DATA_DIR.
for path in save_tables(tables, notes, sources, DATA_DIR):
    print(path.relative_to(REPO) if path.is_relative_to(REPO) else path)